In [ ]:
# Install required libraries (if not already installed)
!pip install pandas numpy scikit-learn

# Import libraries
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 0   # Fake news
true["label"] = 1   # Real news

data = pd.concat([fake, true], axis=0)
data = data.sample(frac=1).reset_index(drop=True)

data.head()

In [ ]:
def clean_text(text):

    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub("\\W"," ",text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)

    return text

In [ ]:
data["text"] = data["text"].apply(clean_text)

In [ ]:
X = data["text"]
y = data["label"]

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.7)

x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

In [ ]:
model = LogisticRegression()
model.fit(x_train_vec, y_train)

In [ ]:
pred = model.predict(x_test_vec)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
def predict_news(news):

    news = clean_text(news)
    vector = vectorizer.transform([news])
    prediction = model.predict(vector)

    if prediction[0] == 0:
        return "Fake News"
    else:
        return "Real News"

In [ ]:
news = input("Enter a news headline or article: ")

print(predict_news(news))

In [ ]:
import pickle

# Save model
with open("fake_news_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Model saved successfully!")

In [ ]:
from google.colab import files

files.download("fake_news_model.pkl")
files.download("vectorizer.pkl")

In [ ]:

files.download("vectorizer.pkl")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

plt.bar(["Precision", "Recall", "F1 Score"], [precision, recall, f1])
plt.title("Model Performance Metrics")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

y_prob = model.predict_proba(x_test_vec)[:,1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()